In [ ]:
import librosa
import librosa.display as dsp
from IPython.display import Audio
import pandas as pd
import numpy as np
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

In [ ]:
# 1. Quay về thư mục gốc của Colab để đảm bảo vị trí đúng
%cd /content

# 2. Xóa thư mục dự án cũ đi nếu nó tồn tại (đây chính là bước "ghi đè")
PROJECT_DIR = 'project_dir'
if os.path.exists(PROJECT_DIR):
    print(f"Thư mục '{PROJECT_DIR}' đã tồn tại. Đang xóa để ghi đè...")
    !rm -rf {PROJECT_DIR}

# 3. Tạo lại thư mục dự án và di chuyển vào đó
print(f"Tạo thư mục mới '{PROJECT_DIR}'...")
!mkdir {PROJECT_DIR}
%cd {PROJECT_DIR}

# 4. Khởi tạo git và tải code bằng sparse checkout
print("Đang tải code từ GitHub...")
!git init
!git remote add origin https://github.com/ChiThanh512/Machine-Learning-Project-251---CEML1.git
!git config core.sparsecheckout true
!echo "HPMR/" >> .git/info/sparse-checkout
!git pull origin main

# 5. Kiểm tra kết quả
print("\n--- Cấu trúc thư mục sau khi tải: ---")
!ls -R

In [ ]:
# Thêm đường dẫn đến thư mục HPMR vào sys.path
hpmr_path = os.path.abspath('HPMR')
if hpmr_path not in sys.path:
    sys.path.append(hpmr_path)
    print(f"Đã thêm '{hpmr_path}' vào sys.path")

# Import hàm download từ module data_loader
from modules.data_loader import download_kaggle_dataset
from modules.preprocessing import read_dataset_and_save_feture
!pip install hmmlearn

In [ ]:
# --- Cấu hình ---
KAGGLE_USER = 'nguyenk512'
KAGGLE_API_KEY = '187454a718c857637f7319f39e33b509'
DATASET_TO_DOWNLOAD = 'subhajournal/free-spoken-digit-database'
TARGET_DIRECTORY = './HPMR/spoken_digit_data' # Thư mục sẽ được tạo bên trong project_dir

# --- Gọi hàm đã import ---
download_kaggle_dataset(
    dataset_name=DATASET_TO_DOWNLOAD,
    username=KAGGLE_USER,
    key=KAGGLE_API_KEY,
    download_dir=TARGET_DIRECTORY
)

In [ ]:
# Đường dẫn đến thư mục dữ liệu đã tải về
DATA_FOLDER = './HPMR/spoken_digit_data' 

# Đường dẫn để lưu file .npz kết quả vào thư mục 'features'
SAVE_FILE_PATH_NPZ = './HPMR/features/processed_data.npz'

# Gọi hàm để bắt đầu quá trình
read_dataset_and_save_feture(root_folder_path=DATA_FOLDER, save_path=SAVE_FILE_PATH_NPZ)

# 1 EDA

In [ ]:
## Định nghĩa các tham số cơ bản
SR = 22050 #tần số lấy mẫu
N_FFT = 512 #int(0.025*SR) # khoảng lấy mẫu fft 25ms
N_HOP = 256 #int(0.010*SR) # bước nhảy giữa 2 frame
N_MFCC = 13
N_MELS =40
pre_emphasis = 0.95

In [ ]:
from modules.EDA import *
df = create_dataframe_from_folders(DATA_FOLDER)
df.head()

In [ ]:
get_random_audio(df,0)

In [ ]:
get_random_audio(df,1)

In [ ]:
get_random_audio(df,2)

In [ ]:
get_random_audio(df,3)

In [ ]:
get_random_audio(df,4)

In [ ]:
get_random_audio(df,5)

In [ ]:
get_random_audio(df,6)

In [ ]:
get_random_audio(df,7)

In [ ]:
get_random_audio(df,8)

In [ ]:
get_random_audio(df,9)

In [ ]:
get_audio_spectogram(df=df)

In [ ]:
get_random_audio(df)

In [ ]:
from modules.preprocessing import load_and_preprocess_data
from modules.preprocessing import split_train_test
 
# Đường dẫn đến file dữ liệu
DATA_FILE_PATH_NPZ = './HPMR/features/processed_data.npz'

# 1. Tải và tiền xử lý
X_processed, y_processed, class_names = load_and_preprocess_data(DATA_FILE_PATH_NPZ)

# 2. Chia dữ liệu
X_train, X_test, y_train, y_test = split_train_test(X_processed, y_processed)

In [ ]:
from modules.training import train_and_evaluate_continue_hmm
import os
import pickle

# 3. Huấn luyện và đánh giá mô hình
models, y_pred, metrics = train_and_evaluate_continue_hmm(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    class_names=class_names,
    num_states=5,
    n_loop=30,
    tol=1e-3
)

# 4. Lưu mô hình trực tiếp
model_save_path = './HPMR/models/continue_hmm.pkl'
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)

# Tạo payload và lưu
payload = []
for m in models:
    payload.append({
        "A": m.A,
        "pi": m.pi,
        "means": m.means,
        "covariances": m.covariances,
    })

with open(model_save_path, "wb") as f:
    pickle.dump({"models": payload, "class_names": class_names}, f)

print(f"\n✅ Model đã được lưu tại: {model_save_path}")
print(f"📊 Tổng kết:")
print(f"   - Accuracy: {metrics['accuracy']:.4f}")
print(f"   - Precision (Macro): {metrics['precision_macro']:.4f}")
print(f"   - Recall (Macro): {metrics['recall_macro']:.4f}")
print(f"   - F1-Score (Macro): {metrics['f1_macro']:.4f}")

In [ ]:
import pickle
import numpy as np
import librosa
from modules.HMM import continueHMM
from modules.preprocessing import extract_features
from sklearn.preprocessing import StandardScaler
from IPython.display import Audio, display

# ===== 1. Tải model đã lưu =====
model_path = './HPMR/models/continue_hmm.pkl'

with open(model_path, 'rb') as f:
    data = pickle.load(f)

models_loaded = []
for p in data["models"]:
    m = continueHMM(A=p["A"], means=p["means"],
                    covariances=p["covariances"], pi=p["pi"])
    models_loaded.append(m)

class_names_loaded = data["class_names"]
print(f"✅ Đã load {len(models_loaded)} models: {class_names_loaded}")

# ===== 2. Khởi tạo scaler từ dữ liệu train (giống cell trước) =====
# Fit scaler trên toàn bộ X_train
X_train_concat = np.vstack(X_train)
scaler = StandardScaler()
scaler.fit(X_train_concat)
print(f"✅ Đã khởi tạo scaler từ {X_train_concat.shape[0]} frames train")

# ===== 3. Upload file WAV từ máy tính (Google Colab) =====
from google.colab import files
print("\n📂 Vui lòng upload file WAV của bạn:")
uploaded = files.upload()

# Lấy tên file đầu tiên được upload
audio_file = list(uploaded.keys())[0]
print(f"\n🎵 File được upload: {audio_file}")

# ===== 4. Trích xuất features (dùng hàm extract_features từ preprocessing) =====
mfcc_features = extract_features(audio_file)

if mfcc_features is None:
    print("❌ Không thể trích xuất features từ file này!")
else:
    print(f"📊 Shape của MFCC features (raw): {mfcc_features.shape}")

    # Chuẩn hóa bằng scaler đã fit trên train set
    mfcc_features_scaled = scaler.transform(mfcc_features)
    print(f"📊 Shape sau chuẩn hóa: {mfcc_features_scaled.shape}")

    # Đọc audio để hiển thị
    y, sr = librosa.load(audio_file, sr=22050)
    print("\n🔊 Phát audio:")
    display(Audio(y, rate=sr))

    # ===== 5. Dự đoán bằng HMM =====
    log_probs = []
    for i, model in enumerate(models_loaded):
        log_prob = model.forward(mfcc_features_scaled)[0]
        log_probs.append(log_prob)
        print(f"   Class {class_names_loaded[i]}: log_prob = {log_prob:.2f}")

    # Chọn class có log_prob cao nhất
    predicted_idx = int(np.argmax(log_probs))
    predicted_class = class_names_loaded[predicted_idx]

    # Tính confidence (xác suất tương đối)
    log_probs_arr = np.array(log_probs)
    probs_normalized = np.exp(log_probs_arr - np.max(log_probs_arr))
    probs_normalized /= probs_normalized.sum()
    confidence = probs_normalized[predicted_idx]

    # ===== 6. Hiển thị kết quả =====
    print("\n" + "="*60)
    print(f"🎯 KẾT QUẢ DỰ ĐOÁN")
    print("="*60)
    print(f"   Số dự đoán: {predicted_class}")
    print(f"   Độ tin cậy: {confidence:.2%}")
    print("="*60)

    # Hiển thị xác suất của tất cả các class
    print("\n📊 Phân phối xác suất:")
    for i, class_name in enumerate(class_names_loaded):
        bar = "█" * int(probs_normalized[i] * 50)
        print(f"   {class_name}: {probs_normalized[i]:>6.1%} {bar}")

    # ===== 7. Vẽ biểu đồ MFCC =====
    import matplotlib.pyplot as plt
    import librosa.display as dsp

    plt.figure(figsize=(12, 6))
    plt.subplot(2, 1, 1)
    dsp.waveshow(y, sr=sr, alpha=0.6)
    plt.title(f'Waveform - Predicted: {predicted_class} (Confidence: {confidence:.1%})')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')

    plt.subplot(2, 1, 2)
    dsp.specshow(mfcc_features_scaled.T, sr=sr, hop_length=256, x_axis='time', cmap='coolwarm')
    plt.colorbar(format='%+2.0f')
    plt.title('MFCC Features (Normalized)')
    plt.xlabel('Time (s)')
    plt.ylabel('MFCC Coefficients')
    plt.tight_layout()
    plt.show()